<a href="https://colab.research.google.com/github/NMinarecioglu/kizilirmak-drought-propagation/blob/main/Sequential_MK_Analysis_%E2%80%94_Basin_Average.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Sequential Mann-Kendall Analysis — Basin Average
9 PNG: SPI-3, SPI-6, SPI-12, SDI-3, SDI-6, SDI-12, RDI-3, RDI-6, RDI-12

Changes from previous version:
  - Basin Average: arithmetic mean of all 5 stations
  - MK statistics box added inside figure
  - Significance band shaded (±1.96)
  - Change point marked with vertical dashed line + year label
  - All labels in English
  - UB calculated correctly (no negation)

Requirements:
    pip install pandas numpy matplotlib pymannkendall openpyxl
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pymannkendall as mk
import os, warnings
warnings.filterwarnings("ignore")

# ============================================================
# SETTINGS
# ============================================================
STATION_FILES = {
    "Kastamonu" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kastamonu_output_indices",
    "Sivas"     : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Sivas_output_indices",
    "Kayseri"   : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kayseri_output_indices",
    "Yozgat"    : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Yozgat_output_indices",
    "Kirikkale" : "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/Kirikkale_output_indices",
}
DATE_COL   = "Date"
OUTPUT_DIR = "/content/drive/MyDrive/Kizilirmak_Analiz_Sonuclari/SQMK_Figures_Claude_Ozet_300DPI"

ALL_INDICES = [
    "SPI-3", "SPI-6", "SPI-12",
    "SDI-3", "SDI-6", "SDI-12",
    "RDI-3", "RDI-6", "RDI-12",
]
# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ----------------------------------------------------------
# Load one station folder or file
# ----------------------------------------------------------
def load_station(path, date_col):
    if os.path.isdir(path):
        files = sorted([
            os.path.join(path, f) for f in os.listdir(path)
            if f.endswith((".xlsx", ".xls", ".csv")) and not f.startswith("~")
        ])
        if not files:
            raise FileNotFoundError(f"No file found in '{path}'.")
        seen = set(); frames = []
        for fp in files:
            ext = os.path.splitext(fp)[1].lower()
            tmp = pd.read_excel(fp, parse_dates=[date_col]) \
                  if ext in [".xlsx", ".xls"] \
                  else pd.read_csv(fp, parse_dates=[date_col])
            new_cols = [date_col] + [c for c in tmp.columns
                                     if c != date_col and c not in seen]
            tmp = tmp[new_cols]; seen.update(tmp.columns)
            frames.append(tmp)
        df = frames[0]
        for f in frames[1:]:
            df = pd.merge(df, f, on=date_col, how="outer")
    else:
        ext = os.path.splitext(path)[1].lower()
        df  = pd.read_excel(path, parse_dates=[date_col]) \
              if ext in [".xlsx", ".xls"] \
              else pd.read_csv(path, parse_dates=[date_col])
    return df.sort_values(date_col).reset_index(drop=True)


# ----------------------------------------------------------
# Compute Basin Average annual mean for a given index
# ----------------------------------------------------------
def basin_average(index_col):
    annual_frames = []
    for station, path in STATION_FILES.items():
        try:
            df = load_station(path, DATE_COL)
        except Exception as e:
            print(f"  [SKIP] {station}: {e}")
            continue
        if index_col not in df.columns:
            continue
        s = (df.set_index(DATE_COL)[index_col]
               .resample("YE").mean()
               .dropna()
               .rename(station))
        annual_frames.append(s)

    if not annual_frames:
        return None, None

    combined = pd.concat(annual_frames, axis=1)
    basin    = combined.mean(axis=1).dropna()
    years    = basin.index.year.values
    values   = basin.values
    return years, values


# ----------------------------------------------------------
# Sequential MK — correct UB formulation
# ----------------------------------------------------------
def sequential_mk(x):
    n  = len(x)
    # UF forward
    uf = np.zeros(n)
    for i in range(1, n):
        s     = sum(np.sign(x[i] - x[j]) for j in range(i))
        e_s   = i*(i-1)/4.0
        var_s = i*(i-1)*(2*i+5)/72.0
        uf[i] = (s - e_s) / np.sqrt(var_s) if var_s > 0 else 0.0

    # UB backward — run UF on reversed series then reverse back (no negation)
    x_rev  = x[::-1]
    ub_rev = np.zeros(n)
    for i in range(1, n):
        s     = sum(np.sign(x_rev[i] - x_rev[j]) for j in range(i))
        e_s   = i*(i-1)/4.0
        var_s = i*(i-1)*(2*i+5)/72.0
        ub_rev[i] = (s - e_s) / np.sqrt(var_s) if var_s > 0 else 0.0
    ub = ub_rev[::-1]

    return uf, ub


# ----------------------------------------------------------
# Find change point (first UF/UB intersection)
# ----------------------------------------------------------
def find_changepoint(uf, ub, years):
    for i in range(1, len(uf)-1):
        if (uf[i] - ub[i]) * (uf[i+1] - ub[i+1]) <= 0:
            return years[i]
    return None


# ----------------------------------------------------------
# Plot one index
# ----------------------------------------------------------
def plot_smk(index_col):
    print(f">>> {index_col}")
    years, values = basin_average(index_col)

    if years is None or len(values) < 5:
        print(f"  [SKIP] Insufficient data for {index_col}.\n")
        return

    n = len(values)

    # Sequential MK
    uf, ub = sequential_mk(values)

    # pymannkendall overall test
    result = mk.original_test(values)
    sig_label = "**" if result.p < 0.01 else \
                ("*"  if result.p < 0.05 else "ns")
    trend_label = result.trend.capitalize()

    print(f"  Trend: {trend_label}  Z={result.z:+.3f}  "
          f"p={result.p:.4f}{sig_label}  slope={result.slope:.5f}")

    # Change point
    cp_year = find_changepoint(uf, ub, years)
    if cp_year:
        print(f"  Change point: {cp_year}")

    # ── Color palette ──────────────────────────────────────
    if index_col.startswith("SPI"):
        bar_pos, bar_neg = "#1f6f2e", "#74c476"
        line_col         = "navy"
    elif index_col.startswith("SDI"):
        bar_pos, bar_neg = "#084594", "#6baed6"
        line_col         = "#084594"
    else:   # RDI
        bar_pos, bar_neg = "#8c2d04", "#fc8d59"
        line_col         = "#8c2d04"

    bar_colors = [bar_pos if v >= 0 else bar_neg for v in values]
    sig_lvl    = 1.96

    # ── Figure ─────────────────────────────────────────────
    fig, ax1 = plt.subplots(figsize=(14, 6))
    fig.patch.set_facecolor("white")

    # Left axis: annual bar chart
    ax1.bar(years, values, color=bar_colors, alpha=0.82,
            width=0.75, zorder=2)
    ax1.axhline(0, color="black", linewidth=0.8, zorder=3)
    ax1.set_ylabel(f"{index_col} Value", color="black", fontsize=11)
    ax1.tick_params(axis="y", labelcolor="black")
    ax1.set_xlabel("Year", fontsize=11)
    ax1.set_xlim(years[0] - 0.8, years[-1] + 0.8)
    ymax = max(abs(values.min()), abs(values.max())) * 1.30
    ax1.set_ylim(-ymax, ymax)
    ax1.grid(axis="y", linestyle="--", alpha=0.3, zorder=1)

    # Right axis: UF / UB
    ax2 = ax1.twinx()

    mk_range = max(abs(uf).max(), abs(ub).max(), sig_lvl + 0.5) * 1.25

    # Shade significance band
    ax2.fill_between([years[0]-1, years[-1]+1],
                     -sig_lvl, sig_lvl,
                     color="lightyellow", alpha=0.45, zorder=1)

    ax2.plot(years, uf, color=line_col,  linewidth=2.0,
             linestyle="-",  label="UF (Forward)", zorder=5)
    ax2.plot(years, ub, color="darkred", linewidth=1.8,
             linestyle="--", label="UB (Backward)", zorder=5)
    ax2.axhline( sig_lvl, color="black", linewidth=1.0,
                linestyle=":", label="0.05 Significance", zorder=4)
    ax2.axhline(-sig_lvl, color="black", linewidth=1.0,
                linestyle=":", zorder=4)

    # UF/UB intersection markers
    for i in range(1, n):
        d_prev = uf[i-1] - ub[i-1]
        d_curr = uf[i]   - ub[i]
        if d_prev * d_curr < 0:
            t  = abs(d_prev) / (abs(d_prev) + abs(d_curr))
            xi = years[i-1] + t * (years[i] - years[i-1])
            yi = uf[i-1]    + t * (uf[i]    - uf[i-1])
            ax2.plot(xi, yi, "ko", markersize=5, zorder=6)

    # Change point vertical line
    if cp_year:
        ax2.axvline(cp_year, color="darkorange", linewidth=1.5,
                    linestyle="-.", alpha=0.9, zorder=6)
        ax2.text(cp_year + 0.4, mk_range * 0.88,
                 f"CP: {cp_year}", fontsize=8.5,
                 color="darkorange", fontweight="bold")

    ax2.set_ylim(-mk_range, mk_range)
    ax2.set_ylabel("MK Test Statistic (u)", color="navy", fontsize=11)
    ax2.tick_params(axis="y", labelcolor="navy")
    ax2.spines["right"].set_edgecolor("navy")
    ax2.spines["right"].set_linewidth(1.2)

    # ── MK statistics box (bottom-right inside figure) ─────
    sig_symbol = ("p < 0.01" if result.p < 0.01 else
                  ("p < 0.05" if result.p < 0.05 else
                   f"p = {result.p:.3f}"))
    stats_text = (
        f"Trend: {trend_label} ({sig_label})\n"
        f"Z = {result.z:+.3f}   {sig_symbol}\n"
        f"Kendall τ = {result.Tau:+.3f}\n"
        f"Sen's slope = {result.slope:+.5f}"
    )
    ax1.text(0.99, 0.03, stats_text,
             transform=ax1.transAxes,
             ha="right", va="bottom", fontsize=8.5,
             color="#1a1a1a",
             bbox=dict(boxstyle="round,pad=0.4",
                       facecolor="white", alpha=0.90,
                       edgecolor="lightgray", linewidth=0.8))

    # ── Title ──────────────────────────────────────────────
    plt.title(
        f"Sequential Mann-Kendall Analysis — {index_col} (Kızılırmak Basin Average)",
        fontsize=12, fontweight="bold", pad=10
    )

    # ── Legend ─────────────────────────────────────────────
    p1 = mpatches.Patch(color=bar_pos, alpha=0.82,
                         label=f"{index_col} Value (≥ 0)")
    p2 = mpatches.Patch(color=bar_neg, alpha=0.82,
                         label=f"{index_col} Value (< 0)")
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(handles=[p1, p2] + h2,
               loc="upper left", fontsize=8.5, framealpha=0.90)

    plt.tight_layout()

    out_path = os.path.join(OUTPUT_DIR, f"Figure4_SQMK_{index_col}.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"  Saved → {out_path}\n")


# ----------------------------------------------------------
# Main loop
# ----------------------------------------------------------
print("=" * 55)
print(" Sequential MK Analysis (Basin Average)")
print("=" * 55 + "\n")

for col in ALL_INDICES:
    plot_smk(col)

print("All 9 figures completed!")
print(f"Output → {OUTPUT_DIR}")